# 2D Polynomial Sky Subtraction — JWST MIRI i2d Mosaics

Removes residual large-scale background gradient from Stage 3 mosaics.

## 1. Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════

# ─── Target ───────────────────────────────────────────────
FILT           = 'F1130W'
BASE_DIR       = '/Users/melyajou/SMC_GO5952/miri'

# ─── Input / Output ──────────────────────────────────────
stage3         = f'{BASE_DIR}/{FILT}/stage3'
INPUT_FILE     = f'{stage3}/miri_{FILT}_final_i2d.fits'
OUTPUT_FILE    = None    # auto: *_skysub.fits
SEGM_FILE      = f'{stage3}/miri_{FILT}_final_segm.fits'

# ─── Sky subtraction parameters ──────────────────────────
POLY_DEGREE    = 2       # 2 = quadratic (6 terms)
SIGMA_UPPER    = 1.5     # sigma clipping threshold — tune per filter
GROW_NPIX      = 5       # mask dilation (pixels) — tune per filter
EDGE_CROP      = 50      # mask mosaic edges (pixels)

# ─── Figures ─────────────────────────────────────────────
CMAP           = 'afmhot'
FIG_DPI        = 200
fig_dir        = f'{BASE_DIR}/{FILT}/figures'

import os
os.makedirs(fig_dir, exist_ok=True)


## 2. Setup

In [ ]:
# ═══════════════════════════════════════════════════════════
# SETUP
# ═══════════════════════════════════════════════════════════
import sys, os, glob
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from astropy.visualization import simple_norm
from scipy.ndimage import binary_erosion

sys.path.insert(0, '..')
from pipeline_utils import *
from skysub_utils import *

## 3. Load and inspect

In [ ]:
# ═══════════════════════════════════════════════════════════
# Load and inspect
# ═══════════════════════════════════════════════════════════
data, header = load_i2d(INPUT_FILE)
print(f'{FILT}: {data.shape}')

## 4. Source masking

In [ ]:
# ═══════════════════════════════════════════════════════════
# Source masking
# ═══════════════════════════════════════════════════════════
# Mask mosaic edges — the mosaic has irregular borders from dithering.
# binary_dilation (in make_source_mask): GROWS the source mask outward
# to catch faint wings around bright sources  using grow_npix
# binary_erosion shrinks any arbitrary shape inward by EDGE_CROP pixels,
# which correctly identifies the border strip regardless of the mosaic geometry.
mask, mean_bg, std_bg = make_source_mask(
    data, segm_file=SEGM_FILE, nsigma=SIGMA_UPPER,
    grow_npix=GROW_NPIX, edge_crop=EDGE_CROP)

In [ ]:
plot_mask_overlay(data, mask, FILT, cmap=CMAP, fig_dir=fig_dir, dpi=FIG_DPI)

## 5. Fit 2D polynomial

In [ ]:
bg_model, coeffs = fit_2d_polynomial(data, mask, degree=POLY_DEGREE)

In [ ]:
plot_bg_model(data, bg_model, FILT, fig_dir=fig_dir, dpi=FIG_DPI)

## 6. Subtract

In [ ]:
data_sub = data - bg_model
data_sub[np.isnan(data)] = np.nan

good_sub = ~np.isnan(data_sub) & ~mask
print(f'Background after subtraction:')
print(f'  mean = {np.nanmean(data_sub[good_sub]):.6e}')
print(f'  std  = {np.nanstd(data_sub[good_sub]):.3f} MJy/sr')

## 7. Before / After

In [ ]:
plot_before_after(data, data_sub, FILT, poly_degree=POLY_DEGREE,
                  cmap=CMAP, fig_dir=fig_dir, dpi=FIG_DPI)

## 8. Background histogram

In [ ]:
plot_background_histogram(data_sub, good_sub, FILT, fig_dir=fig_dir, dpi=FIG_DPI)

## 9. Save

In [ ]:
# Save
OUTPUT_FILE = save_skysub(INPUT_FILE, data_sub, bg_model, mask, header,
                          POLY_DEGREE, SIGMA_UPPER, GROW_NPIX, SEGM_FILE)

## 10. Uncertainties

In [ ]:
pixel_noise, bkg_uncertainty, poly_bias = compute_uncertainties(data_sub, good_sub)

## 11. Summary

In [ ]:
print_summary(FILT, data_sub, good_sub, pixel_noise, bkg_uncertainty, poly_bias,
              POLY_DEGREE, SIGMA_UPPER, GROW_NPIX, EDGE_CROP, SEGM_FILE, OUTPUT_FILE)